# Train EfficientNetV2-S on Colab (3 seeds)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Run the cells in order. Each training cell is one seed (~15–30 min). Keep this tab open while it runs.

**If the session disconnects:** run cells 1–4 again, then continue with the next seed. Finished seeds are skipped automatically, because their results are already on Google Drive.

## 1. Connect Google Drive
Checkpoints and logs are saved here, so they survive when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Get the code and data
Clones the repo the first time; pulls the latest code if it's already there.

In [ ]:
import os

REPO = "/content/COMP-6936_Project"
if os.path.exists(REPO):
    %cd {REPO}
    !git pull
else:
    !git clone https://github.com/labib-islam/COMP-6936_Project.git {REPO}
    %cd {REPO}

!git log --oneline -1

## 3. Check the GPU
Should print `True` and `Tesla T4`. If it says `False`, change the runtime type to T4 GPU and start again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 4. Settings
These must be the same for every model. Only `MODEL` differs between the four notebooks.

`OUT` is the shared Drive folder. If the folder was shared with you, open it in Drive and choose **Organize → Add shortcut → My Drive**, so it appears at this path.

In [ ]:
MODEL = "efficientnet_v2_s"
LR = 1e-4
EPOCHS = 15
PATIENCE = 5
OUT = "/content/drive/MyDrive/brisc_checkpoints/runs"

os.makedirs(OUT, exist_ok=True)


def run_dir(seed):
    return f"{OUT}/{MODEL}_lr{LR:g}_seed{seed}"


def is_done(seed):
    # a run is finished when its summary.json exists
    return os.path.exists(f"{run_dir(seed)}/summary.json")


for seed in [42, 43, 44]:
    print(seed, "done" if is_done(seed) else "not done", run_dir(seed))

## 5. Train seed 42

In [ ]:
SEED = 42
if is_done(SEED):
    print("Already finished:", run_dir(SEED))
else:
    !python -u -m src.training.train --model {MODEL} --lr {LR} --seed {SEED} --epochs {EPOCHS} --patience {PATIENCE} --amp --out {OUT}

## 6. Train seed 43

In [ ]:
SEED = 43
if is_done(SEED):
    print("Already finished:", run_dir(SEED))
else:
    !python -u -m src.training.train --model {MODEL} --lr {LR} --seed {SEED} --epochs {EPOCHS} --patience {PATIENCE} --amp --out {OUT}

## 7. Train seed 44

In [ ]:
SEED = 44
if is_done(SEED):
    print("Already finished:", run_dir(SEED))
else:
    !python -u -m src.training.train --model {MODEL} --lr {LR} --seed {SEED} --epochs {EPOCHS} --patience {PATIENCE} --amp --out {OUT}

## 8. Results for this model
Reads the three `summary.json` files from Drive.

In [ ]:
import json
import pandas as pd

rows = []
for seed in [42, 43, 44]:
    path = f"{run_dir(seed)}/summary.json"
    if not os.path.exists(path):
        print("missing:", path)
        continue
    with open(path) as f:
        s = json.load(f)
    rows.append({
        "seed": seed,
        "best_val_f1": s["best_val_f1"],
        "best_epoch": s["best_epoch"],
        "epochs_run": s["epochs_run"],
        "minutes": s["total_minutes"],
        "peak_memory_mb": s["peak_memory_mb"],
        "amp": s["settings"]["amp"],
        "device": s["device"],
    })

results = pd.DataFrame(rows)
print(results.round(4).to_string(index=False))
if len(results) > 1:
    print(f"\nval macro-F1: {results.best_val_f1.mean():.4f} ± {results.best_val_f1.std():.4f}")

## 9. After all three seeds
- The run folders (with `best.pt`, `log.csv`, `summary.json`) are in `MyDrive/brisc_checkpoints/runs/`.
- Download each run's `log.csv` and `summary.json` into the local `runs/` folder of the repo and commit them. Leave `best.pt` on Drive (it's gitignored).